# Piloto de comorbilidades para ARMD-AI

## Objetivo

Este notebook deja documentada la evaluacion inicial del archivo de comorbilidades y muestra resultados concretos ya guardados en CSV.

## Que se hizo fuera de memoria completa

El archivo `microbiology_cultures_comorbidity.csv` pesa cerca de `20 GB`, por lo que no se cargo completo en RAM.

La revision se hizo por streaming y luego se guardaron artefactos resumen para leerlos aqui.


In [4]:
from collections import Counter, defaultdict
import csv
from pathlib import Path
import pandas as pd

RUTA_PROYECTO = Path.cwd().resolve().parent
RUTA_PROCESADOS = RUTA_PROYECTO / "3.DATOS-PROCESADOS"
RUTA_DATA = RUTA_PROYECTO / "data"
RUTA_BASE_LIMPIA = RUTA_PROCESADOS / "armd_s_aureus_base_limpia.csv"
RUTA_COMORB = RUTA_DATA / "microbiology_cultures_comorbidity.csv"

RUTA_RESUMEN = RUTA_PROCESADOS / "resumen_comorbilidades_piloto.csv"
RUTA_TOP = RUTA_PROCESADOS / "top_comorbilidades_piloto.csv"

print(RUTA_RESUMEN)
print(RUTA_TOP)
print(RUTA_COMORB)


D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\3.DATOS-PROCESADOS\resumen_comorbilidades_piloto.csv
D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\3.DATOS-PROCESADOS\top_comorbilidades_piloto.csv
D:\6TO-SEMESTRE\Sistemas Inteligentes\2doCorte\Proyecto\modelo\data\microbiology_cultures_comorbidity.csv


## Resumen cuantitativo principal

Esta tabla resume cobertura, granularidad y faltantes del archivo de comorbilidades al cruzarlo con la cohorte limpia de `S. aureus`.


In [5]:
if not RUTA_RESUMEN.exists() or not RUTA_TOP.exists():
    df_base = pd.read_csv(RUTA_BASE_LIMPIA, usecols=["anon_id", "order_proc_id_coded"])
    df_base["order_proc_id_coded"] = df_base["order_proc_id_coded"].astype(str)
    ordenes_objetivo = set(df_base["order_proc_id_coded"].unique())
    orden_a_paciente = (
        df_base.drop_duplicates(subset=["order_proc_id_coded"])
        .set_index("order_proc_id_coded")["anon_id"]
        .astype(str)
        .to_dict()
    )

    total_filas_match = 0
    ordenes_con_match = set()
    pacientes_con_match = set()
    componentes_unicos = set()
    contador_componentes = Counter()
    componentes_por_orden = defaultdict(set)
    nulos_end = 0
    nulos_start = 0

    with RUTA_COMORB.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            orden = str(row.get("order_proc_id_coded"))
            if orden not in ordenes_objetivo:
                continue

            total_filas_match += 1
            ordenes_con_match.add(orden)
            paciente = orden_a_paciente.get(orden)
            if paciente is not None:
                pacientes_con_match.add(paciente)

            componente = row.get("comorbidity_component")
            if componente:
                componentes_unicos.add(componente)
                contador_componentes[componente] += 1
                componentes_por_orden[orden].add(componente)

            start_val = row.get("comorbidity_component_start_days_culture")
            end_val = row.get("comorbidity_component_end_days_culture")
            if start_val in (None, ""):
                nulos_start += 1
            if end_val in (None, ""):
                nulos_end += 1

    serie_componentes_orden = pd.Series([len(v) for v in componentes_por_orden.values()], dtype="float64")
    total_ordenes_base = len(ordenes_objetivo)

    df_resumen = pd.DataFrame(
        [
            {"metrica": "ordenes_totales_s_aureus", "valor": total_ordenes_base},
            {"metrica": "ordenes_con_comorbilidades", "valor": len(ordenes_con_match)},
            {"metrica": "porcentaje_ordenes_con_comorbilidades", "valor": round(len(ordenes_con_match) * 100 / total_ordenes_base, 2)},
            {"metrica": "pacientes_con_comorbilidades", "valor": len(pacientes_con_match)},
            {"metrica": "filas_coincidentes_comorbilidades", "valor": total_filas_match},
            {"metrica": "componentes_unicos", "valor": len(componentes_unicos)},
            {"metrica": "promedio_componentes_unicos_por_orden", "valor": round(float(serie_componentes_orden.mean()), 2) if not serie_componentes_orden.empty else 0.0},
            {"metrica": "percentil_95_componentes_unicos_por_orden", "valor": int(serie_componentes_orden.quantile(0.95)) if not serie_componentes_orden.empty else 0},
            {"metrica": "maximo_componentes_unicos_por_orden", "valor": int(serie_componentes_orden.max()) if not serie_componentes_orden.empty else 0},
            {"metrica": "porcentaje_nulos_start_days", "valor": round(nulos_start * 100 / total_filas_match, 2) if total_filas_match else 0.0},
            {"metrica": "porcentaje_nulos_end_days", "valor": round(nulos_end * 100 / total_filas_match, 2) if total_filas_match else 0.0},
        ]
    )
    df_top = pd.DataFrame(contador_componentes.most_common(20), columns=["comorbidity_component", "frecuencia"])
    df_resumen.to_csv(RUTA_RESUMEN, index=False)
    df_top.to_csv(RUTA_TOP, index=False)
    print("Artefactos de comorbilidades regenerados desde notebook.")

df_resumen = pd.read_csv(RUTA_RESUMEN)
display(df_resumen)


Artefactos de comorbilidades regenerados desde notebook.


,metrica,valor
0,ordenes_totales_s_aureus,8414.00
1,ordenes_con_comorbilidades,8342.00
2,porcentaje_ordenes_con_comorbilidades,99.14
3,pacientes_con_comorbilidades,4855.00
4,filas_coincidentes_comorbilidades,2474947.00
5,componentes_unicos,498.00
6,promedio_componentes_unicos_por_orden,30.99
7,percentil_95_componentes_unicos_por_orden,86.00
8,maximo_componentes_unicos_por_orden,182.00
9,porcentaje_nulos_start_days,0.00


## Top de componentes mas frecuentes

Esta tabla permite ver rapidamente que el archivo mezcla componentes clinicos utiles con categorias potencialmente ambiguas o ruidosas.


In [6]:
df_top = pd.read_csv(RUTA_TOP)
display(df_top)


,comorbidity_component,frecuencia
0,Congestive heart failure,407766
1,Cystic fibrosis,256153
2,Sinusitis,74740
3,Abnormal findings without diagnosis,59459
4,Other specified status,51215
5,Organ transplant status,40321
6,Other specified and unspecified gastrointestin...,36742
7,Diabetes mellitus without complication,34299
8,Respiratory signs and symptoms,34182
9,"Implant, device or graft related encounter",30231


## Interpretacion

### Señales favorables
- la cobertura sobre la cohorte es alta
- muchas ordenes si tienen informacion de comorbilidades
- algunos componentes si parecen clinicamente valiosos para predecir susceptibilidad

### Riesgos detectados
- hay casi `500` componentes distintos
- el promedio de componentes por orden es muy alto
- algunas ordenes llegan a mas de `180` componentes unicos
- `end_days` esta casi siempre vacio
- aparecen categorias demasiado generales o sospechosas como `Invalid PDX`, `Other specified status` o `Abnormal findings without diagnosis`


## Conclusion metodologica

El archivo de comorbilidades:

- si puede aportar valor clinico
- pero no es apto para entrar crudo al modelo

La estrategia piloto correcta, si se quiere probar en el futuro, seria:

1. seleccionar solo `10` a `15` comorbilidades o grupos auditados
2. excluir categorias ruidosas o demasiado generales
3. construir un `multihot` reducido por `order_proc_id_coded`
4. no usar por ahora `end_days`


## Relacion con los CSV de entrenamiento

Despues de construir las variantes con comorbilidades, quedan `5` datasets de entrenamiento/modelado:

- `armd_s_aureus_base_modelado_base.csv`
- `armd_s_aureus_base_modelado_ampliada.csv`
- `armd_s_aureus_base_modelado_multihot_abx.csv`
- `armd_s_aureus_base_modelado_multihot_comorb.csv`
- `armd_s_aureus_base_modelado_multihot_abx_comorb.csv`

La construccion ejecutable de las dos nuevas variantes queda documentada en `4.MODELADO-Y-VALIDACION/construccion_datasets_comorb.ipynb`.
